# SAS to Databricks: Advanced Regression with Feature Engineering

This notebook builds on the beginner lesson with multiple input columns, a train-test split, feature scaling, and a PySpark ML pipeline.


## 1. Import Libraries

The code uses clear names and keeps each pipeline stage visible so newer Python users can follow the flow.


In [ ]:
import warnings

import numpy as np
import pandas as pd
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import PolynomialExpansion, StandardScaler, VectorAssembler
from pyspark.ml.regression import LinearRegression
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

print("Libraries are ready")
print(f"Spark version: {spark.version}")


## 2. Create a Multi-Feature Dataset

A realistic model usually uses more than one input column. This example predicts house price from property features.


In [ ]:
np.random.seed(42)
row_count = 500

house_data = pd.DataFrame({
    "square_footage": np.random.uniform(1000, 6000, row_count),
    "bedrooms": np.random.randint(1, 6, row_count),
    "bathrooms": np.random.uniform(1, 4, row_count),
    "age": np.random.randint(0, 100, row_count),
    "garage_spaces": np.random.randint(0, 4, row_count),
    "has_pool": np.random.choice([0, 1], row_count),
    "lot_size_sqft": np.random.uniform(5000, 50000, row_count),
})

house_data["price"] = (
    120 * house_data["square_footage"]
    + 45000 * house_data["bedrooms"]
    + 30000 * house_data["bathrooms"]
    - 1500 * house_data["age"]
    + 15000 * house_data["garage_spaces"]
    + 75000 * house_data["has_pool"]
    + 2 * house_data["lot_size_sqft"]
    + np.random.normal(0, 50000, row_count)
)
house_data["price"] = np.maximum(house_data["price"], 100000)

houses = spark.createDataFrame(house_data)

print(f"Rows: {houses.count()}")
houses.show(5)


## 3. SAS Reference: Multi-Feature Data

SAS stores all model columns in one dataset. PySpark stores the same columns in a Spark DataFrame.

```sas
DATA house_data;
  DO row_id = 1 TO 500;
    square_footage = 1000 + RAND('UNIFORM') * 5000;
    bedrooms = CEIL(RAND('UNIFORM') * 5);
    bathrooms = 1 + RAND('UNIFORM') * 3;
    age = FLOOR(RAND('UNIFORM') * 100);
    garage_spaces = FLOOR(RAND('UNIFORM') * 4);
    has_pool = FLOOR(RAND('UNIFORM') * 2);
    lot_size_sqft = 5000 + RAND('UNIFORM') * 45000;
    price = 120*square_footage + 45000*bedrooms + 30000*bathrooms - 1500*age;
    OUTPUT;
  END;
RUN;
```


## 4. Visualize Feature Relationships with Matplotlib

Matplotlib can show how one feature relates to the target. This chart also uses color, so learners can see that a third variable can be included visually.


In [ ]:
plot_data = houses.select("square_footage", "age", "price").limit(300).toPandas()

plt.figure(figsize=(8, 5))
scatter = plt.scatter(
    plot_data["square_footage"],
    plot_data["price"],
    c=plot_data["age"],
    alpha=0.75,
)
plt.title("House Price by Square Footage and Age")
plt.xlabel("Square Footage")
plt.ylabel("Price")
plt.colorbar(scatter, label="Age")
plt.grid(True)
plt.show()


## 5. Split Data into Train and Test Sets

Train data teaches the model. Test data checks how well the model works on rows it has not seen.


In [ ]:
train_data, test_data = houses.randomSplit([0.7, 0.3], seed=42)

feature_columns = [
    "square_footage",
    "bedrooms",
    "bathrooms",
    "age",
    "garage_spaces",
    "has_pool",
    "lot_size_sqft",
]
target_column = "price"

train_count = train_data.count()
test_count = test_data.count()
total_count = train_count + test_count

print(f"Train rows: {train_count} ({train_count / total_count:.1%})")
print(f"Test rows: {test_count} ({test_count / total_count:.1%})")


## 6. SAS Reference: Train-Test Split

SAS commonly creates a random number and splits rows with an `IF` statement. PySpark uses `randomSplit`.

```sas
DATA house_split;
  SET house_data;
  random_value = RAND('UNIFORM');
  IF random_value < 0.70 THEN split = 'train';
  ELSE split = 'test';
RUN;

DATA train_data test_data;
  SET house_split;
  IF split = 'train' THEN OUTPUT train_data;
  ELSE OUTPUT test_data;
RUN;
```


## 7. Build a Pipeline

A pipeline keeps the feature steps and the model together. This is easier to reuse than running each step by hand.


In [ ]:
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")
scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withMean=True, withStd=True)
regression = LinearRegression(featuresCol="scaled_features", labelCol=target_column)

linear_pipeline = Pipeline(stages=[assembler, scaler, regression])

print("Training linear pipeline...")
linear_model = linear_pipeline.fit(train_data)
print("Training complete")

regression_stage = linear_model.stages[-1]

print("\nModel coefficients")
for column_name, coefficient in zip(feature_columns, regression_stage.coefficients):
    print(f"{column_name:<18} {coefficient:>12,.2f}")
print(f"{'intercept':<18} {regression_stage.intercept:>12,.2f}")


## 8. SAS Reference: Scaling and Regression

SAS often standardizes numeric variables with `PROC STANDARD`, then trains with `PROC REG`. PySpark combines those steps in a reusable `Pipeline`.

```sas
PROC STANDARD DATA=train_data MEAN=0 STD=1 OUT=train_scaled;
  VAR square_footage bedrooms bathrooms age garage_spaces has_pool lot_size_sqft;
RUN;

PROC REG DATA=train_scaled;
  MODEL price = square_footage bedrooms bathrooms age garage_spaces has_pool lot_size_sqft;
RUN;
QUIT;
```


## 9. Evaluate the Model

Comparing train and test metrics helps reveal overfitting. If test error is much worse than train error, the model may not generalize well.


In [ ]:
train_predictions = linear_model.transform(train_data)
test_predictions = linear_model.transform(test_data)

rmse_evaluator = RegressionEvaluator(labelCol=target_column, predictionCol="prediction", metricName="rmse")
mae_evaluator = RegressionEvaluator(labelCol=target_column, predictionCol="prediction", metricName="mae")
r2_evaluator = RegressionEvaluator(labelCol=target_column, predictionCol="prediction", metricName="r2")

train_rmse = rmse_evaluator.evaluate(train_predictions)
test_rmse = rmse_evaluator.evaluate(test_predictions)
train_mae = mae_evaluator.evaluate(train_predictions)
test_mae = mae_evaluator.evaluate(test_predictions)
train_r2 = r2_evaluator.evaluate(train_predictions)
test_r2 = r2_evaluator.evaluate(test_predictions)

print(f"{'Metric':<10} {'Train':>15} {'Test':>15}")
print("-" * 42)
print(f"{'RMSE':<10} {train_rmse:>15,.0f} {test_rmse:>15,.0f}")
print(f"{'MAE':<10} {train_mae:>15,.0f} {test_mae:>15,.0f}")
print(f"{'R2':<10} {train_r2:>15.4f} {test_r2:>15.4f}")


## 10. Visualize Actual vs Predicted Prices

In this chart, each dot compares one real price with the model prediction. Dots close to the diagonal line are better predictions.


In [ ]:
actual_vs_predicted = test_predictions.select(target_column, "prediction").toPandas()

min_price = min(actual_vs_predicted[target_column].min(), actual_vs_predicted["prediction"].min())
max_price = max(actual_vs_predicted[target_column].max(), actual_vs_predicted["prediction"].max())

plt.figure(figsize=(6, 6))
plt.scatter(actual_vs_predicted[target_column], actual_vs_predicted["prediction"], alpha=0.65)
plt.plot([min_price, max_price], [min_price, max_price], color="red", label="Perfect prediction")
plt.title("Actual vs Predicted Test Prices")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.legend()
plt.grid(True)
plt.show()


## 11. Try Polynomial Features

Polynomial features add interactions and curved terms. They can improve fit, but they also make the model more complex.


In [ ]:
polynomial_assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")
polynomial_expansion = PolynomialExpansion(degree=2, inputCol="features", outputCol="polynomial_features")
polynomial_scaler = StandardScaler(inputCol="polynomial_features", outputCol="scaled_polynomial_features", withMean=True, withStd=True)
polynomial_regression = LinearRegression(featuresCol="scaled_polynomial_features", labelCol=target_column)

polynomial_pipeline = Pipeline(stages=[
    polynomial_assembler,
    polynomial_expansion,
    polynomial_scaler,
    polynomial_regression,
])

print("Training polynomial pipeline...")
polynomial_model = polynomial_pipeline.fit(train_data)
print("Training complete")

polynomial_train_predictions = polynomial_model.transform(train_data)
polynomial_test_predictions = polynomial_model.transform(test_data)

polynomial_train_r2 = r2_evaluator.evaluate(polynomial_train_predictions)
polynomial_test_r2 = r2_evaluator.evaluate(polynomial_test_predictions)
polynomial_test_rmse = rmse_evaluator.evaluate(polynomial_test_predictions)

print(f"{'Model':<18} {'Train R2':>12} {'Test R2':>12} {'Test RMSE':>15}")
print("-" * 62)
print(f"{'Linear':<18} {train_r2:>12.4f} {test_r2:>12.4f} {test_rmse:>15,.0f}")
print(f"{'Polynomial':<18} {polynomial_train_r2:>12.4f} {polynomial_test_r2:>12.4f} {polynomial_test_rmse:>15,.0f}")


## 12. Visualize Train and Test R2 Scores

R2 is a fit score, so taller bars are better. Comparing train and test bars helps learners spot overfitting.


In [ ]:
model_names = ["Linear Train", "Linear Test", "Polynomial Train", "Polynomial Test"]
r2_scores = [train_r2, test_r2, polynomial_train_r2, polynomial_test_r2]

plt.figure(figsize=(9, 5))
plt.bar(model_names, r2_scores)
plt.title("R2 Score Comparison")
plt.ylabel("R2 Score")
plt.ylim(0, 1.05)
plt.xticks(rotation=20)
plt.grid(axis="y")
plt.show()


## 13. Migration Notes

| Step | SAS | Databricks |
| --- | --- | --- |
| Standardize features | `PROC STANDARD` | `StandardScaler` |
| Multiple regression | `PROC REG` | `LinearRegression` |
| Train-test split | `PROC SURVEYSELECT` or manual split | `randomSplit` |
| Repeatable workflow | Macro or stored process | `Pipeline` |
| Score new data | `PROC SCORE` | `pipeline_model.transform()` |

Keep the linear model as the baseline. Use the polynomial model only if it improves test results enough to justify the added complexity.
